# 3-1 OpenCV Test

- `models/three_model_compare_weighted/resnet18_best.pt` 체크포인트를 불러옵니다.
- 웹캠 프레임마다 클래스별 확률을 OpenCV 화면에 표시합니다.
- 얼굴이 잡히면 얼굴 영역으로 추론하고, 못 잡으면 전체 프레임으로 추론합니다.


In [12]:
from pathlib import Path
import time
from collections import Counter, deque

import cv2
import numpy as np
import torch
from PIL import Image
from torchvision import models, transforms

checkpoint_path = Path("models/three_model_compare_weighted/resnet18_best.pt")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

sample_interval = 0.2
window_seconds = 3.0
confidence_threshold = 0.70
state_ratio_threshold = 0.80
device


device(type='cuda')

In [13]:
checkpoint = torch.load(checkpoint_path, map_location=device)
class_names = checkpoint["class_names"]
image_size = checkpoint.get("image_size", 224)
freeze_backbone = checkpoint.get("freeze_backbone", True)

print("class_names:", class_names)
print("image_size:", image_size)
print("freeze_backbone:", freeze_backbone)


class_names: ['Attentive', 'Drowsy', 'LookingAway']
image_size: 224
freeze_backbone: True


In [14]:
def build_resnet18_model(class_count, freeze_backbone=True):
    model = models.resnet18(weights=None)

    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False

    model.fc = torch.nn.Linear(model.fc.in_features, class_count)
    return model


model = build_resnet18_model(len(class_names), freeze_backbone=freeze_backbone)
model.load_state_dict(checkpoint["model_state"])
model = model.to(device)
model.eval()

transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)


In [15]:
def get_face_crop(frame_bgr, margin=0.35):
    gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)

    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=5,
        minSize=(80, 80)
    )

    if len(faces) == 0:
        return None, None

    x, y, w, h = max(faces, key=lambda box: box[2] * box[3])

    H, W = frame_bgr.shape[:2]

    mx = int(w * margin)
    my = int(h * margin)

    x1 = max(0, x - mx)
    y1 = max(0, y - my)
    x2 = min(W, x + w + mx)
    y2 = min(H, y + h + my)

    crop_bgr = frame_bgr[y1:y2, x1:x2]

    return crop_bgr, (x1, y1, x2 - x1, y2 - y1)


def predict_single_frame(frame_bgr):
    crop_bgr, face_box = get_face_crop(frame_bgr)

    if crop_bgr is None:
        return "No Face", None, None, 0.0

    rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
    pil_image = Image.fromarray(rgb)

    x = transform(pil_image).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        logits = model(x)
        probs = torch.softmax(logits, dim=1)[0].cpu().numpy()

    pred_idx = int(np.argmax(probs))
    pred_label = class_names[pred_idx]
    max_prob = float(probs[pred_idx])

    return pred_label, probs, face_box, max_prob


prediction_buffer = deque()
last_sample_time = 0.0


def update_probability_buffer(pred_label, probs):
    now = time.time()

    if probs is None:
        return

    prediction_buffer.append((now, pred_label, probs))

    while prediction_buffer and now - prediction_buffer[0][0] > window_seconds:
        prediction_buffer.popleft()


def get_stable_prediction():
    if len(prediction_buffer) == 0:
        return "Waiting", None, 0.0, 0.0

    labels = [item[1] for item in prediction_buffer]
    probs_list = [item[2] for item in prediction_buffer]

    mean_probs = np.mean(probs_list, axis=0)

    stable_idx = int(np.argmax(mean_probs))
    stable_label = class_names[stable_idx]
    stable_prob = float(mean_probs[stable_idx])

    label_counter = Counter(labels)
    label_ratio = label_counter[stable_label] / len(labels)

    if stable_prob < confidence_threshold:
        return "Uncertain", mean_probs, stable_prob, label_ratio

    if label_ratio < state_ratio_threshold:
        return "Waiting", mean_probs, stable_prob, label_ratio

    return stable_label, mean_probs, stable_prob, label_ratio


In [16]:
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    raise RuntimeError("웹캠을 열 수 없습니다.")

last_raw_label = "Waiting"
last_max_prob = 0.0
last_face_box = None
last_mean_probs = None
stable_label = "Waiting"
stable_prob = 0.0
stable_ratio = 0.0
last_sample_time = 0.0

try:
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame = cv2.flip(frame, 1)

        now = time.time()

        if now - last_sample_time >= sample_interval:
            last_sample_time = now

            raw_label, probs, face_box, max_prob = predict_single_frame(frame)

            last_raw_label = raw_label
            last_max_prob = max_prob
            last_face_box = face_box

            if probs is not None:
                update_probability_buffer(raw_label, probs)

            stable_label, mean_probs, stable_prob, stable_ratio = get_stable_prediction()
            last_mean_probs = mean_probs

        if last_face_box is not None:
            x, y, w, h = last_face_box
            cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)

        cv2.putText(
            frame,
            f"raw: {last_raw_label} ({last_max_prob * 100:.1f}%)",
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (255, 255, 255),
            2
        )

        cv2.putText(
            frame,
            f"stable: {stable_label} ({stable_prob * 100:.1f}%, ratio {stable_ratio * 100:.1f}%)",
            (20, 75),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 255, 0),
            2
        )

        if last_mean_probs is not None:
            y0 = 115
            for i, class_name in enumerate(class_names):
                prob_text = f"{class_name}: {last_mean_probs[i] * 100:.1f}%"
                cv2.putText(
                    frame,
                    prob_text,
                    (20, y0 + i * 30),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.7,
                    (255, 255, 0),
                    2
                )

        cv2.imshow("OpenCV Stable Inference", frame)

        key = cv2.waitKey(1) & 0xFF
        if key == ord("q"):
            break
finally:
    cap.release()
    cv2.destroyAllWindows()
